<a href="https://colab.research.google.com/github/JWasonga/Statistical_Data_Analytics/blob/Applied_Statistics_and_Econometrics-_Portfolio/2_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Mushroom Edibility - Data Cleaning & Feature Engineering

Pipeline:

   1. Encode `class` (e->0, p->1)
   2. Replace '?' in `stalk_root` with a new "missing" category (preserves the missingness signal)
   3. Drop the constant columns(s) (`veil_type`)
   4. One-hot encode all remaining categorical features (`drop_first=True`)
   5. Save to `data/mushrooms_cleaned.csv`

#1. Imports & Load Raw Data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append(".")
from utils import (load_data, encode_target, handle_missing, drop_constant, preprocess_data)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

%matplotlib inline

In [3]:
df = load_data("mushrooms.csv")
print(f"Raw Shape: {df.shape}")
df.head()

Raw Shape: (8124, 23)


,class,cap_shape,cap_surface,cap_color,bruises,odor,gill_attachment,gill_spacing,gill_size,gill_color,stalk_shape,stalk_root,stalk_surface_above_ring,stalk_surface_below_ring,stalk_color_above_ring,stalk_color_below_ring,veil_type,veil_color,ring_number,ring_type,spore_print_color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,e,e,s,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,e,c,s,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,e,c,s,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,e,e,s,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,t,e,s,s,w,w,p,w,o,e,n,a,g


#2. Missing / Invalid Values

The UCI dataset uses `'?'` to mark missing `stalk_root` values

In [4]:
missing_marker = (df == "?").sum()
print(missing_marker[missing_marker > 0])
print(f"\nDuplicate rows:  {df.duplicated().sum()}")

stalk_root    2480
dtype: int64

Duplicate rows:  0


#3. Encode Target

In [5]:
df_enc = encode_target(df)
print("Class distribution:")
print(df_enc["class"].value_counts())
df_enc.head()

Class distribution:
class
0    4208
1    3916
Name: count, dtype: int64


,class,cap_shape,cap_surface,cap_color,bruises,odor,gill_attachment,gill_spacing,gill_size,gill_color,stalk_shape,stalk_root,stalk_surface_above_ring,stalk_surface_below_ring,stalk_color_above_ring,stalk_color_below_ring,veil_type,veil_color,ring_number,ring_type,spore_print_color,population,habitat
0,1,x,s,n,t,p,f,c,n,k,e,e,s,s,w,w,p,w,o,p,k,s,u
1,0,x,s,y,t,a,f,c,b,k,e,c,s,s,w,w,p,w,o,p,n,n,g
2,0,b,s,w,t,l,f,c,b,n,e,c,s,s,w,w,p,w,o,p,n,n,m
3,1,x,y,w,t,p,f,c,n,n,e,e,s,s,w,w,p,w,o,p,k,s,u
4,0,x,s,g,f,n,f,w,b,k,t,e,s,s,w,w,p,w,o,e,n,a,g


#4. Handle Missing Values

We treat missing `stalk_root` as it's own category (`m`issing`) rather than imputing - the missingness itself may carry signal.

In [6]:
df_filled = handle_missing(df_enc, strategy="category")
print("After replacing '?' with 'missing' :")
print(df_filled["stalk_root"].value_counts())

After replacing '?' with 'missing' :
stalk_root
b          3776
missing    2480
e          1120
c           556
r           192
Name: count, dtype: int64


#5. Drop Constant Columns

In [7]:
df_no_const, dropped = drop_constant(df_filled)
print(f"Dropped columns (constant): {dropped}")
print(f"Shape after dropping constant columns: {df_no_const.shape}")

Dropped columns (constant): ['veil_type']
Shape after dropping constant columns: (8124, 22)


#6. Distribution Check Pre/Post Encoding

In [8]:
cat_cols = [c for c in df_no_const.columns if c != "class" and df_no_const[c].dtype == object]
print(f" Will one-hot encode  {len(cat_cols)} categorical columns")
print(f"   -> total dummy columns:",
      sum(df_no_const[c].nunique() - 1 for c in cat_cols))

 Will one-hot encode  21 categorical columns
   -> total dummy columns: 95


#7. Run the Full Pipeline

In [9]:
df_processed = preprocess_data(df, missing_strategy="category")
print(f"Processed shape: {df_processed.shape}")
print(f"Missing values: {df_processed.isnull().sum().sum()}")
df_processed.head()

Processed shape: (8124, 96)
Missing values: 0


,class,cap_shape_c,cap_shape_f,cap_shape_k,cap_shape_s,cap_shape_x,cap_surface_g,cap_surface_s,cap_surface_y,cap_color_c,cap_color_e,cap_color_g,cap_color_n,cap_color_p,cap_color_r,cap_color_u,cap_color_w,cap_color_y,bruises_t,odor_c,odor_f,odor_l,odor_m,odor_n,odor_p,odor_s,odor_y,gill_attachment_f,gill_spacing_w,gill_size_n,gill_color_e,gill_color_g,gill_color_h,gill_color_k,gill_color_n,gill_color_o,gill_color_p,gill_color_r,gill_color_u,gill_color_w,...,stalk_color_above_ring_o,stalk_color_above_ring_p,stalk_color_above_ring_w,stalk_color_above_ring_y,stalk_color_below_ring_c,stalk_color_below_ring_e,stalk_color_below_ring_g,stalk_color_below_ring_n,stalk_color_below_ring_o,stalk_color_below_ring_p,stalk_color_below_ring_w,stalk_color_below_ring_y,veil_color_o,veil_color_w,veil_color_y,ring_number_o,ring_number_t,ring_type_f,ring_type_l,ring_type_n,ring_type_p,spore_print_color_h,spore_print_color_k,spore_print_color_n,spore_print_color_o,spore_print_color_r,spore_print_color_u,spore_print_color_w,spore_print_color_y,population_c,population_n,population_s,population_v,population_y,habitat_g,habitat_l,habitat_m,habitat_p,habitat_u,habitat_w
0,1,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0
1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
3,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0
4,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0


#8. Sanity Checks & Save

In [10]:
assert df_processed["class"].isin([0, 1]).all()
assert df_processed.isnull().sum().sum() == 0
assert "veil_type" not in df_processed.columns
print("All checks passed.")

All checks passed.


In [11]:
df_processed.to_csv("mushrroms_cleaned.csv", index=False)
print(f"Saved to data/mushrooms_cleaned.csv ({df_processed.shape[0]} rows, {df_processed.shape[1]} cols)")

Saved to data/mushrooms_cleaned.csv (8124 rows, 96 cols)
